# Fine-tune a fast NLI model on FEVER

Warm-starts from `cross-encoder/nli-MiniLM2-L6-H768` (fast, standard attention — already used by the PBEL backend) and continues training it on FEVER's claim/evidence pairs, so it keeps the same inference speed but learns FEVER-style judgments (the DeBERTa-v3 FEVER checkpoint we benchmarked earlier had this skill but was ~19x slower on CPU).

**Setup**: Runtime → Change runtime type → Hardware accelerator → GPU (T4 is fine, free tier). Then Runtime → Run all.

In [ ]:
!pip install -q transformers datasets accelerate evaluate scikit-learn

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (go enable the GPU runtime!)')

## Load FEVER NLI data

`pietrolesci/nli_fever` on the Hugging Face Hub is FEVER already reshaped into (premise=claim, hypothesis=evidence, label=entailment/neutral/contradiction) triples — exactly what an NLI classifier needs.

In [ ]:
from datasets import load_dataset

raw = load_dataset('pietrolesci/nli_fever')
print(raw)
label_names = raw['train'].features['label'].names
print('dataset label names (by id):', label_names)

## Load the base model (warm start)

We continue training the exact checkpoint already deployed, so the fine-tuned model keeps its speed characteristics — only the weights change, not the architecture.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

BASE_MODEL = 'cross-encoder/nli-MiniLM2-L6-H768'

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL)

model_id2label = {k: v.lower() for k, v in model.config.id2label.items()}
model_label2id = {v: k for k, v in model_id2label.items()}
print('model label scheme:', model_id2label)

# Don't assume the dataset's label ids line up with the model's — map by name.
dataset_id_to_model_id = {
    i: model_label2id[name.lower()] for i, name in enumerate(label_names)
}
print('dataset id -> model id remap:', dataset_id_to_model_id)

In [ ]:
def tokenize_and_remap(batch):
    enc = tokenizer(batch['premise'], batch['hypothesis'], truncation=True, max_length=128)
    enc['labels'] = [dataset_id_to_model_id[l] for l in batch['label']]
    return enc

cols_to_remove = raw['train'].column_names
train_ds = raw['train'].map(tokenize_and_remap, batched=True, remove_columns=cols_to_remove)
eval_ds = raw['dev'].map(tokenize_and_remap, batched=True, remove_columns=cols_to_remove)
print(train_ds)
print(eval_ds)

## Train

MiniLM-L6 is small (~23M params), so on a free T4 this should take roughly 30-60 minutes for 2 epochs over the full ~208k train pairs. Reduce `num_train_epochs` or subsample `train_ds` (e.g. `train_ds.select(range(50000))`) if you want a faster/cheaper run.

In [ ]:
import numpy as np
import evaluate
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments

accuracy = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=preds, references=labels)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

args = TrainingArguments(
    output_dir='./fever-minilm-nli',
    per_device_train_batch_size=64,
    per_device_eval_batch_size=128,
    num_train_epochs=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    fp16=torch.cuda.is_available(),
    logging_steps=200,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
print(trainer.evaluate())

## Sanity-check on our known failure case

The deployed MiniLM checkpoint got this wrong (said SUPPORTED). Confirm the fine-tuned model calls it correctly before deploying.

In [ ]:
model.eval()
test_pairs = [
    ('Modern flat Earth beliefs', 'The Earth is flat.'),
    ('The Earth is a sphere that orbits the Sun.', 'The Earth is flat.'),
]
inputs = tokenizer([p[0] for p in test_pairs], [p[1] for p in test_pairs], return_tensors='pt', truncation=True, padding=True).to(model.device)
with torch.no_grad():
    probs = torch.softmax(model(**inputs).logits, dim=-1)
for (premise, hyp), p in zip(test_pairs, probs):
    scores = {model_id2label[i]: round(float(p[i]), 3) for i in range(len(p))}
    print(f'[{premise!r} -> {hyp!r}]  {scores}')

## Save and export

**Option A (recommended): push to the Hugging Face Hub.** Then the VM just needs `NLI_MODEL=<your-username>/fever-minilm-nli` as an env var — `verify.py` already loads whatever `NLI_MODEL` points to via `from_pretrained`, so no file transfer is needed. Get a token (with write access) from https://huggingface.co/settings/tokens.

In [ ]:
from huggingface_hub import login
login()  # paste your HF token when prompted

REPO_ID = 'your-username/fever-minilm-nli'  # change this
model.push_to_hub(REPO_ID, private=True)
tokenizer.push_to_hub(REPO_ID, private=True)

**Option B: download a zip instead**, if you'd rather not use the Hub. You'd then `scp` it to the VM and point `NLI_MODEL` at the local directory path.

In [ ]:
model.save_pretrained('./fever-minilm-nli-final')
tokenizer.save_pretrained('./fever-minilm-nli-final')

import shutil
shutil.make_archive('fever-minilm-nli-final', 'zip', './fever-minilm-nli-final')

from google.colab import files
files.download('fever-minilm-nli-final.zip')